In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lxml
import html5lib
%matplotlib inline
from urllib.request import urlopen
import time

pd.set_option('display.max_columns',1000)
pd.set_option('display.max_rows',1000)

In [2]:
df = pd.read_csv('data/df_mlb2.csv')

/var/folders/y0/_yc3t8mn3kx8j1td1w609rl00000gn/T/ipykernel_77598/2372536421.py:1: DtypeWarning: Columns (72,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/df_mlb2.csv')


In [3]:
oddsshark_num_to_team_dict = {
    26995 + i: team for i, team in enumerate([
        'PHI', 'SDN', 'SFN', 'ANA', 'DET', 'CIN', 'NYA', 'TEX', 'TBA', 'COL',
        'MIN', 'KCA', 'ARI', 'BAL', 'ATL', 'TOR', 'SEA', 'MIL', 'PIT', 'NYN',
        'LAN', 'OAK', 'WAS', 'CHA', 'SLN', 'CHN', 'BOS', 'MIA', 'HOU', 'CLE'
    ])
}

In [10]:
# Use the saved files to get the odds information
# We create a dict based on team and season for easy lookup
df_odds_dict={}
for i in range(26995, 27025):
    team_name = oddsshark_num_to_team_dict[i]
    df_odds_dict[team_name] = {}
    print(team_name)
    for season in range(2021,2025):
        fname = 'data/odds/oddsshark_'+team_name+'_'+str(season)+'.csv'
        df_temp = pd.read_csv(fname)
        df_temp['date_dblhead'] = (df_temp.date_numeric.astype(str) + df_temp.dblheader_num.astype(str)).astype(int)
        df_temp.set_index('date_dblhead', inplace=True)
        df_odds_dict[team_name][season] = df_temp

PHI
SDN
SFN
ANA
DET
CIN
NYA
TEX
TBA
COL
MIN
KCA
ARI
BAL
ATL
TOR
SEA
MIL
PIT
NYN
LAN
OAK
WAS
CHA
SLN
CHN
BOS
MIA
HOU
CLE


In [11]:
# Again, we iterate through the main dataframe
# get the team, season, game and then get
# the relevant info from the odds dictionary

implied_prob_h = np.zeros(df.shape[0])
implied_prob_v = np.zeros(df.shape[0])
over_under = np.zeros(df.shape[0])
ou_result = np.full(df.shape[0],'', dtype=object)
for ind, row in df.iterrows():
    if (ind%1000)==0:
        print(ind)
    if row.season<2021:
        continue
    else:
        season = row['season']
        home_team = row['team_h']
        visit_team = row['team_v']
        date_dblh = row['date_dblhead']
        try:
            implied_prob_h[ind] = df_odds_dict[home_team][season].loc[date_dblh,'prob_implied']
            over_under[ind] = df_odds_dict[home_team][season].loc[date_dblh,'Total']
            ou_result[ind] = df_odds_dict[home_team][season].loc[date_dblh,'OU']
        except KeyError:
            print(f'Game not found wrt home_team:{home_team} vs {visit_team} date_dbl {date_dblh}')
        try:
            implied_prob_v[ind] = df_odds_dict[visit_team][season].loc[date_dblh,'prob_implied']
        except KeyError:
            print(f'Game not found wrt visit_team:{visit_team} vs {home_team} date_dbl {date_dblh}')

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
Game not found wrt home_team:CHA vs SEA date_dbl 202106260
Game not found wrt visit_team:SEA vs CHA date_dbl 202106260
Game not found wrt home_team:ATL vs SDN date_dbl 202107211
Game not found wrt visit_team:SDN vs ATL date_dbl 202107211
Game not found wrt home_team:ATL vs SDN date_dbl 202107212
Game not found wrt visit_team:SDN vs ATL date_dbl 202107212
93000
94000
95000
96000
97000
98000
Game not found wrt home_team:NYN vs MIA date_dbl 202309280
Gam

In [12]:
df['implied_prob_h'] = implied_prob_h
df['implied_prob_v'] = implied_prob_v
df['implied_prob_h_mid'] = (implied_prob_h + (1-implied_prob_v))/2
df['over_under_line']=over_under
df['over_under_result']=ou_result

In [13]:
df[(df.season>=2021) & (df.implied_prob_h==0)]

,day_of_week,team_v,league_v,team_h,league_h,runs_v,runs_h,num_outs,day_night,park_id,attendance,duration,AB_v,H_v,x2B_v,x3B_v,HR_v,RBI_v,sac_hits_v,sac_flies_v,HBP_v,BB_v,IBB_v,K_v,SB_v,CS_v,GDP_v,CI_v,LOB_v,num_pitchers_v,IER_v,ER_v,WP_v,balks_v,PO_v,assists_v,ERR_v,PB_v,x2P_v,x3P_v,AB_h,H_h,x2B_h,x3B_h,HR_h,RBI_h,sac_hits_h,sac_flies_h,HBP_h,BB_h,IBB_h,K_h,SB_h,CS_h,GDP_h,CI_h,LOB_h,num_pitchers_h,IER_h,ER_h,WP_h,balks_h,PO_h,assists_h,ERR_h,PB_h,x2P_h,x3P_h,ump_home_id,ump_first_id,ump_second_id,ump_third_id,ump_lf_id,ump_rf_id,manager_v,manager_h,starting_pitcher_id_v,starting_pitcher_name_v,starting_pitcher_id_h,starting_pitcher_name_h,batter1_id_v,batter1_name_v,batter1_pos_v,batter2_id_v,batter2_name_v,batter2_pos_v,visiting_2_id.1,batter3_name_v,batter3_pos_v,batter4_id_v,batter4_name_v,batter4_pos_v,batter5_id_v,batter5_name_v,batter5_pos_v,batter6_id_v,batter6_name_v,batter6_pos_v,batter7_id_v,batter7_name_v,batter7_pos_v,batter8_id_v,batter8_name_v,batter8_pos_v,batter9_id_v,batter9_name_v,batter9_pos_v,batter1_id_h,batter1_name_h,batter1_pos_h,batter2_id_h,batter2_name_h,batter2_pos_h,batter3_id_h,batter3_name_h,batter3_pos_h,batter4_id_h,batter4_name_h,batter4_pos_h,batter5_id_h,batter5_name_h,batter5_pos_h,batter6_id_h,batter6_name_h,batter6_pos_h,batter7_id_h,batter7_name_h,batter7_pos_h,batter8_id_h,batter8_name_h,batter8_pos_h,batter9_id_h,batter9_name_h,batter9_pos_h,season,run_diff,home_victory,run_total,date_dblhead,BATAVG_162_h,BATAVG_162_v,BATAVG_90_h,BATAVG_90_v,BATAVG_30_h,BATAVG_30_v,OBP_162_h,OBP_162_v,OBP_90_h,OBP_90_v,OBP_30_h,OBP_30_v,SLG_162_h,SLG_162_v,SLG_90_h,SLG_90_v,SLG_30_h,SLG_30_v,OBS_162_h,OBS_162_v,OBS_90_h,OBS_90_v,OBS_30_h,OBS_30_v,SB_162_h,SB_162_v,SB_90_h,SB_90_v,SB_30_h,SB_30_v,CS_162_h,CS_162_v,CS_90_h,CS_90_v,CS_30_h,CS_30_v,ERR_162_h,ERR_162_v,ERR_90_h,ERR_90_v,ERR_30_h,ERR_30_v,implied_prob_h,implied_prob_v,implied_prob_h_mid,over_under_line,over_under_result
92553,Sat,SEA,AL,CHA,AL,3,2,54,D,CHI12,30017.0,197,34,7,0,0,3,3,0,0,0,3,0,9,0,0,1,0,7,7,2,2,0,0,27,9,0,0,3,0,29,6,0,0,0,2,0,1,1,3,0,11,1,0,3,0,5,3,3,3,0,0,27,7,1,0,1,0,cuzzp901,hallt901,rippm901,blasc901,NaN,NaN,servs002,larut101,gilbl002,Logan Gilbert,lynnl001,Lance Lynn,crawj002,J.P. Crawford,6,hanim001,Mitch Haniger,9,seagk001,Kyle Seager,5,frant002,Ty France,3,bauej001,Jake Bauers,7,longs001,Shed Long,4,torrl001,Luis Torrens,2,fralj001,Jake Fraley,10,tramt001,Taylor Trammell,8,andet001,Tim Anderson,6,goodb001,Brian Goodwin,8,moncy001,Yoan Moncada,5,abrej003,Jose Abreu,3,grany001,Yasmani Grandal,2,lambj001,Jake Lamb,7,mercy001,Yermin Mercedes,10,garcl004,Leury Garcia,4,gonzl005,Luis Gonzalez,9,2021,-1,0,5,202106260,0.261093,0.220639,0.244710,0.214905,0.239378,0.243539,0.327324,0.292443,0.320749,0.283228,0.314019,0.301194,0.433627,0.372102,0.395904,0.367071,0.383420,0.410537,0.760951,0.664545,0.716653,0.650299,0.697438,0.711731,56.0,104.0,38.0,42.0,11.0,15.0,27.0,34.0,17.0,15.0,7.0,5.0,102.0,81.0,59.0,47.0,18.0,15.0,0.0,0.0,0.5,0.0,
92836,Wed,SDN,NL,ATL,NL,3,2,42,D,ATL03,28621.0,156,26,7,2,0,1,3,0,1,0,4,0,6,0,1,0,0,7,3,1,1,0,0,21,10,1,1,0,0,25,5,1,0,0,1,1,1,0,0,0,3,0,0,0,0,4,3,3,3,2,0,21,8,0,0,0,0,timmt901,riggj901,mahrn901,marqa901,NaN,NaN,tingj801,snitb801,paddc001,Chris Paddack,mullk001,Kyle Muller,phamt001,Tommy Pham,7,tatif002,Fernando Tatis,6,cronj001,Jake Cronenworth,3,machm001,Manny Machado,5,myerw001,Wil Myers,9,profj001,Jurickson Profar,8,kim-h002,Ha-Seong Kim,4,rivaw001,Webster Rivas,2,paddc001,Chris Paddack,1,pedej001,Joc Pederson,9,swand001,Dansby Swanson,6,freef001,Freddie Freeman,3,albio001,Ozzie Albies,4,rilea001,Austin Riley,5,hereg002,Guillermo Heredia,8,arcio002,Orlando Arcia,7,smitk002,Kevan Smith,2,mullk001,Kyle Muller,1,2021,-1,0,5,202107211,0.250874,0.248832,0.243334,0.245403,0.248227,0.277168,0.326777,0.323520,0.319782,0.323847,0.326067,0.352100,0.447673,0.429105,0.433682,0.412237,0.427558,0.499501,0.774450,0.752625,0.753464,0.736083,0.753625,0.851602,69.0,146.0,43.0,86.0,19.0,22.0,20.0,40.0,15.0,

In [14]:
indices_to_drop = df[(df.season>=2021) & (df.implied_prob_h==0)].index
indices_to_drop

Int64Index([92553, 92836, 92837, 98651], dtype='int64')

In [15]:
df.shape

(98704, 186)

In [16]:
df.drop(indices_to_drop, inplace=True)
df.shape

(98700, 186)

In [17]:
df.reset_index(inplace=True, drop=True)
df.shape

(98700, 186)

In [19]:
df.to_csv('data/df_mlb3.csv', index=False)